# 04 — Advanced Model Candidate

## Purpose

This notebook develops the next forecasting model after the baseline machine learning workflow.

The goal is to test a stronger production candidate that can outperform the Random Forest benchmark while still remaining practical and interpretable.

Current benchmark to beat:

- Random Forest WMAPE = **0.2195**

This notebook will focus on:

- preparing the same modeling dataset and feature set
- training a stronger tabular forecasting model
- evaluating it against the current benchmark
- checking whether it is a better candidate for production v1

## Scope

This notebook will include:

- loading the controlled modeling subset
- reusing the validated forecasting features
- training the next candidate model
- evaluating WMAPE against prior baselines

This notebook will **not** include:

- MLflow tracking
- model registration
- deployment logic
- batch inference orchestration

Those steps will come after the best notebook candidate is chosen.

In [1]:
# Core
import warnings

# Data
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt

# SQL / project imports
from sqlalchemy import text

from app_config.config import settings
from database.database import warehouse_engine

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 200)

print("Libraries and warehouse connection imports loaded successfully.")
print("Configured warehouse DSN present:", bool(settings.WAREHOUSE_DSN or settings.POSTGRES_DSN))

Libraries and warehouse connection imports loaded successfully.
Configured warehouse DSN present: True


## Load modeling subset

To keep iteration fast and controlled, this notebook will reuse the same **top-20 item-store series subset** used in the previous modeling notebook.

This ensures:

- direct comparability with the Random Forest benchmark
- stable testing conditions across candidate models
- faster experimentation before scaling to the full dataset

The source table remains:

`gold.gold_m5_daily_feature_mart`

In [2]:
subset_query = """
with series_rank as (
    select
        store_id,
        item_id,
        sum(sales) as total_sales
    from gold.gold_m5_daily_feature_mart
    group by store_id, item_id
),
top_series as (
    select
        store_id,
        item_id
    from series_rank
    order by total_sales desc
    limit 20
)
select
    g.id,
    g.item_id,
    g.dept_id,
    g.cat_id,
    g.store_id,
    g.state_id,
    g.d,
    g.date,
    g.wm_yr_wk,
    g.sales,
    g.sell_price,
    g.temperature_2m_max,
    g.temperature_2m_min,
    g.precipitation_sum,
    g.wind_speed_10m_max,
    g.cpi_all_items,
    g.unemployment_rate,
    g.federal_funds_rate,
    g.nonfarm_payrolls,
    g.trends_walmart,
    g.trends_grocery_store,
    g.trends_discount_store,
    g.trends_cleaning_supplies
from gold.gold_m5_daily_feature_mart g
join top_series t
  on g.store_id = t.store_id
 and g.item_id = t.item_id
order by g.store_id, g.item_id, g.date
"""

with warehouse_engine.begin() as conn:
    result = conn.execute(text(subset_query))
    model_df = pd.DataFrame(result.fetchall(), columns=result.keys())

model_df.shape

(1800, 23)

In [3]:
model_df["date"] = pd.to_datetime(model_df["date"])

model_df = model_df.sort_values(["store_id", "item_id", "date"]).reset_index(drop=True)

model_df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,date,wm_yr_wk,sales,sell_price,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,cpi_all_items,unemployment_rate,federal_funds_rate,nonfarm_payrolls,trends_walmart,trends_grocery_store,trends_discount_store,trends_cleaning_supplies
0,FOODS_3_090_CA_1_validation,FOODS_3_090,FOODS_3,FOODS,CA_1,CA,d_1824,2016-01-26,11552,37,1,16.5,2.5,0.0,6.0,237,4,0,143210,49,46,57,62
1,FOODS_3_090_CA_1_validation,FOODS_3_090,FOODS_3,FOODS,CA_1,CA,d_1825,2016-01-27,11552,35,1,16.3,5.7,0.0,7.2,237,4,0,143210,51,46,55,73
2,FOODS_3_090_CA_1_validation,FOODS_3_090,FOODS_3,FOODS,CA_1,CA,d_1826,2016-01-28,11552,38,1,15.5,3.3,0.0,11.2,237,4,0,143210,51,46,49,68
3,FOODS_3_090_CA_1_validation,FOODS_3_090,FOODS_3,FOODS,CA_1,CA,d_1827,2016-01-29,11552,39,1,17.2,7.9,0.0,6.2,237,4,0,143210,52,46,57,76
4,FOODS_3_090_CA_1_validation,FOODS_3_090,FOODS_3,FOODS,CA_1,CA,d_1828,2016-01-30,11601,65,1,11.8,9.5,8.2,9.2,237,4,0,143210,68,57,83,45


In [4]:
# ============================================================
# Feature engineering for the advanced model candidate
# ============================================================
# Purpose:
# - create calendar features from the date column
# - create lag features from historical sales
# - create rolling mean features using only past data
# Notes:
# - all lag/rolling features are grouped by (store_id, item_id)
# - shift(1) is used before rolling windows to keep features leakage-safe
# ============================================================

# Group definition for per-series feature engineering
group_cols = ["store_id", "item_id"]

# ------------------------------------------------------------
# Calendar features
# ------------------------------------------------------------
model_df["day_of_week"] = model_df["date"].dt.dayofweek
model_df["day_of_month"] = model_df["date"].dt.day
model_df["week_of_year"] = model_df["date"].dt.isocalendar().week.astype(int)
model_df["month"] = model_df["date"].dt.month

# ------------------------------------------------------------
# Lag features
# ------------------------------------------------------------
model_df["lag_1"] = model_df.groupby(group_cols)["sales"].shift(1)
model_df["lag_7"] = model_df.groupby(group_cols)["sales"].shift(7)
model_df["lag_28"] = model_df.groupby(group_cols)["sales"].shift(28)

# ------------------------------------------------------------
# Rolling mean features
# ------------------------------------------------------------
# shift(1) ensures the current day's sales are excluded
model_df["rolling_mean_7"] = (
    model_df.groupby(group_cols)["sales"]
    .shift(1)
    .rolling(7)
    .mean()
)

model_df["rolling_mean_28"] = (
    model_df.groupby(group_cols)["sales"]
    .shift(1)
    .rolling(28)
    .mean()
)

# ------------------------------------------------------------
# Preview engineered features
# ------------------------------------------------------------
model_df[
    [
        "store_id",
        "item_id",
        "date",
        "sales",
        "day_of_week",
        "lag_1",
        "lag_7",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_28",
    ]
].head(35)

,store_id,item_id,date,sales,day_of_week,lag_1,lag_7,lag_28,rolling_mean_7,rolling_mean_28
0,CA_1,FOODS_3_090,2016-01-26,37,1,NaN,NaN,NaN,NaN,NaN
1,CA_1,FOODS_3_090,2016-01-27,35,2,37.0,NaN,NaN,NaN,NaN
2,CA_1,FOODS_3_090,2016-01-28,38,3,35.0,NaN,NaN,NaN,NaN
3,CA_1,FOODS_3_090,2016-01-29,39,4,38.0,NaN,NaN,NaN,NaN
4,CA_1,FOODS_3_090,2016-01-30,65,5,39.0,NaN,NaN,NaN,NaN
5,CA_1,FOODS_3_090,2016-01-31,39,6,65.0,NaN,NaN,NaN,NaN
6,CA_1,FOODS_3_090,2016-02-01,51,0,39.0,NaN,NaN,NaN,NaN
7,CA_1,FOODS_3_090,2016-02-02,31,1,51.0,37.0,NaN,43.428571,NaN
8,CA_1,FOODS_3_090,2016-02-03,48,2,31.0,35.0,NaN,42.571429,NaN
9,CA_1,FOODS_3_090,2016-02-04,50,3,48.0,38.0,NaN,44.428571,NaN


In [5]:
# ============================================================
# Prepare modeling dataset for advanced candidate training
# ============================================================
# Purpose:
# - keep only rows where all engineered historical features exist
# - define the feature columns used for model training
# Notes:
# - early rows in each series are dropped because lag/rolling
#   features are not available at the beginning of the series
# ============================================================

# ------------------------------------------------------------
# Feature list for model training
# ------------------------------------------------------------
feature_columns = [
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "lag_1",
    "lag_7",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
]

# ------------------------------------------------------------
# Drop rows with missing engineered features
# ------------------------------------------------------------
modeling_df = model_df.dropna(subset=feature_columns).copy()

# ------------------------------------------------------------
# Preview modeling dataset shape
# ------------------------------------------------------------
print("Modeling dataset shape:", modeling_df.shape)
modeling_df.head()

Modeling dataset shape: (1240, 32)


,id,item_id,dept_id,cat_id,store_id,state_id,d,date,wm_yr_wk,sales,sell_price,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,cpi_all_items,unemployment_rate,federal_funds_rate,nonfarm_payrolls,trends_walmart,trends_grocery_store,trends_discount_store,trends_cleaning_supplies,day_of_week,day_of_month,week_of_year,month,lag_1,lag_7,lag_28,rolling_mean_7,rolling_mean_28
28,FOODS_3_090_CA_1_validation,FOODS_3_090,FOODS_3,FOODS,CA_1,CA,d_1852,2016-02-23,11604,38,1,21.1,5.7,0.0,7.1,237,4,0,143406,50,47,62,78,1,23,8,2,43.0,37.0,37.0,55.142857,47.892857
29,FOODS_3_090_CA_1_validation,FOODS_3_090,FOODS_3,FOODS,CA_1,CA,d_1853,2016-02-24,11604,26,1,22.1,6.6,0.0,9.3,237,4,0,143406,52,47,61,78,2,24,8,2,38.0,46.0,35.0,55.285714,47.928571
30,FOODS_3_090_CA_1_validation,FOODS_3_090,FOODS_3,FOODS,CA_1,CA,d_1854,2016-02-25,11604,30,1,22.4,6.7,0.0,8.1,237,4,0,143406,52,47,65,69,3,25,8,2,26.0,32.0,38.0,52.428571,47.607143
31,FOODS_3_090_CA_1_validation,FOODS_3_090,FOODS_3,FOODS,CA_1,CA,d_1855,2016-02-26,11604,69,1,24.0,7.8,0.0,8.4,237,4,0,143406,55,47,67,75,4,26,8,2,30.0,54.0,39.0,52.142857,47.321429
32,FOODS_3_090_CA_1_validation,FOODS_3_090,FOODS_3,FOODS,CA_1,CA,d_1856,2016-02-27,11605,92,1,21.1,7.9,0.0,9.4,237,4,0,143406,68,68,94,66,5,27,8,2,69.0,74.0,65.0,54.285714,48.392857


In [6]:
# ============================================================
# Create time-based train/test split
# ============================================================
# Purpose:
# - simulate a real forecasting setup
# - reserve the most recent 28 days for evaluation
# Notes:
# - this keeps temporal ordering intact
# - no random split is used in forecasting workflows
# ============================================================

# ------------------------------------------------------------
# Define 28-day holdout window
# ------------------------------------------------------------
split_date = modeling_df["date"].max() - pd.Timedelta(days=27)

# ------------------------------------------------------------
# Split into train and test sets
# ------------------------------------------------------------
train_df = modeling_df[modeling_df["date"] < split_date].copy()
test_df = modeling_df[modeling_df["date"] >= split_date].copy()

# ------------------------------------------------------------
# Separate features and target
# ------------------------------------------------------------
X_train = train_df[feature_columns]
y_train = train_df["sales"]

X_test = test_df[feature_columns]
y_test = test_df["sales"]

# ------------------------------------------------------------
# Print split summary
# ------------------------------------------------------------
print("Split date:", split_date.date())
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Split date: 2016-03-28
Train shape: (680, 32)
Test shape: (560, 32)
X_train shape: (680, 9)
X_test shape: (560, 9)


In [7]:
# ============================================================
# Train the next advanced model candidate
# ============================================================
# Purpose:
# - fit a stronger tabular model than Random Forest
# - use gradient boosting as the next practical production candidate
# Notes:
# - GradientBoostingRegressor is a strong next step because it
#   often performs better on structured/tabular forecasting features
# - it remains much easier to interpret and operationalize than
#   jumping straight into deep learning
# ============================================================

from sklearn.ensemble import GradientBoostingRegressor

# ------------------------------------------------------------
# Initialize model
# ------------------------------------------------------------
gbr_model = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

# ------------------------------------------------------------
# Train model
# ------------------------------------------------------------
gbr_model.fit(X_train, y_train)

print("Gradient Boosting model trained successfully.")

Gradient Boosting model trained successfully.


In [8]:
# ============================================================
# Evaluate the Gradient Boosting model
# ============================================================
# Purpose:
# - generate predictions on the held-out test set
# - compute WMAPE so the result is directly comparable with:
#   * naive lag-1
#   * seasonal naive lag-7
#   * rolling mean baseline
#   * Random Forest
# ============================================================

# ------------------------------------------------------------
# Generate test predictions
# ------------------------------------------------------------
test_df["pred_gbr"] = gbr_model.predict(X_test)

# ------------------------------------------------------------
# Compute WMAPE
# ------------------------------------------------------------
wmape_gbr = np.sum(np.abs(test_df["sales"] - test_df["pred_gbr"])) / np.sum(test_df["sales"])

print("Gradient Boosting WMAPE:", round(wmape_gbr, 4))

Gradient Boosting WMAPE: 0.2303


### Advanced model evaluation: Gradient Boosting

The Gradient Boosting model was evaluated using the same time-based holdout window used for all previous models.

Evaluation result:

**Gradient Boosting WMAPE = 0.2303**

Comparison with previous models:

| Model | WMAPE |
|------|------|
| Naive lag-1 | 0.2893 |
| Rolling mean (7-day) | 0.2621 |
| Seasonal naive lag-7 | 0.2423 |
| Gradient Boosting | 0.2303 |
| Random Forest | **0.2195** |

Observations:

- Gradient Boosting improves significantly over all naive baselines.
- However, it **does not outperform the Random Forest model**.
- The Random Forest currently remains the **best-performing candidate model** in this experimentation stage.

At this point in the workflow, Random Forest remains the leading candidate for the first production model.

## Notebook conclusion

This notebook evaluated a stronger tabular model candidate using Gradient Boosting.

Key results:

- Baseline models confirmed strong weekly seasonality.
- Random Forest achieved the best performance so far.
- Gradient Boosting improved over naive baselines but did not surpass Random Forest.

Current best model:

**Random Forest (WMAPE = 0.2195)**

Next step in the MLOps workflow:

The modeling logic developed in these notebooks will now be **refactored into production training code**. This will include:

- modular feature engineering functions
- reproducible training scripts
- experiment tracking
- model artifact management

The notebooks have served their purpose: **model discovery and validation**. The next stage focuses on **production-grade MLOps implementation**.